# 03: Tensor Formatting for Deep Sequence Learning

**Objective:** Transform 2D contiguous blocks into 3D tensors compatible with Deep Sequence DL engines.

**Key Actions:**
- Strict Longitudinal Scaling: RobustScaler adjusted uniquely on labeled training folds.
- Tensors reshaping: 2D Pandas to 3D NumPy (HCPs, 86, Features).
- Export to .npy artifacts.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import logging
import os

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# silver_df = pd.read_parquet('data/silver/silver_layer.parquet')

## 1. Strict Scaling
Avoid leaking future info or unlabelled data distributions into the scaler parameters.

In [ ]:
def strict_longitudinal_scaling(df: pd.DataFrame, dyn_cols: list, val_fold: int = 0) -> pd.DataFrame:
    """
    Fits RobustScaler on Train subset and transforms all.
    """
    logger.info("Applying strict longitudinal scaling...")
    if df.empty: return df
    
    train_mask = (df['fold'] != -1) & (df['fold'] != val_fold)
    
    scaler = RobustScaler()
    scaler.fit(df.loc[train_mask, dyn_cols])
    
    df_scaled = df.copy()
    df_scaled[dyn_cols] = scaler.transform(df_scaled[dyn_cols])
    
    logger.info("Scaling applied.")
    return df_scaled

# dyn_cols = ['TRX_WEEKLY'] # Add relevant sequence columns
# scaled_df = strict_longitudinal_scaling(silver_df, dyn_cols)

## 2. Converting to Tensors

In [ ]:
def generate_3d_tensor(df: pd.DataFrame, id_col: str, time_col: str, feature_cols: list, seq_len: int = 86):
    """
    Translates 2D dataframe efficiently into 3D tensors.
    """
    logger.info("Reshaping to 3D Tensors...")
    if df.empty: return np.array([]), np.array([])
    
    df = df.sort_values([id_col, time_col])
    unique_ids = df[id_col].unique()
    
    n_entities = len(unique_ids)
    n_features = len(feature_cols)
    
    raw_values = df[feature_cols].values
    tensor_3d = raw_values.reshape((n_entities, seq_len, n_features))
    
    logger.info(f"Tensor generation complete. Shape: {tensor_3d.shape}")
    return tensor_3d, unique_ids

# tensor_3d, unique_ids = generate_3d_tensor(scaled_df, 'NUEVO_ID', 'WEEK_ID', dyn_cols)
# os.makedirs('data/tensors', exist_ok=True)
# np.save('data/tensors/longitudinal_tensor.npy', tensor_3d)
# np.save('data/tensors/tensor_ids.npy', unique_ids)